# 01 — Exceptions: the four-clause `try`, and `raise from`

Exceptions are *objects* that propagate up the call stack until something catches them. They aren't just an error mechanism — they're how Python communicates exceptional outcomes (`StopIteration`, `KeyError`, even `KeyboardInterrupt`).

Three rules of thumb:

1. **Catch specific exceptions, not `Exception`.** Bare `except Exception:` swallows bugs.
2. **Preserve the original cause** with `raise NewError(...) from original`.
3. **Use `finally` for cleanup**, `else` for post-success work.

## The built-in hierarchy (relevant subset)

```
BaseException
 ├── SystemExit
 ├── KeyboardInterrupt
 └── Exception                  # catch this, never BaseException
      ├── ValueError
      ├── TypeError
      ├── KeyError
      ├── IndexError
      ├── AttributeError
      ├── FileNotFoundError     # subclass of OSError
      ├── ZeroDivisionError
      └── ...
```

Rule: **catch `Exception` at most once at the top of your program** (to log unhandled errors). Everywhere else, catch the specific subclass.

In [1]:
# Why specific is better than generic:
def parse_age(s):
    return int(s)

# BAD — swallows everything, including bugs in your own code.
try:
    print(parse_age("not a number"))
except Exception as e:
    print("something went wrong:", e)

# GOOD — only catches what you actually expect.
try:
    print(parse_age("not a number"))
except ValueError as e:
    print("bad number:", e)

something went wrong: invalid literal for int() with base 10: 'not a number'
bad number: invalid literal for int() with base 10: 'not a number'


## The four clauses: `try / except / else / finally`

In [2]:
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError as e:
        print("  except: caught", e)
        return None
    else:
        # Runs ONLY if no exception. Put post-success work here so it's NOT
        # accidentally wrapped in the try block (which would catch its errors too).
        print("  else  : division ok")
        return result
    finally:
        # ALWAYS runs — success path, exception path, even if we return inside try.
        print("  finally: cleanup")

print("-> divide(10, 2)")
print("result:", divide(10, 2))
print()
print("-> divide(10, 0)")
print("result:", divide(10, 0))

-> divide(10, 2)
  else  : division ok
  finally: cleanup
result: 5.0

-> divide(10, 0)
  except: caught division by zero
  finally: cleanup
result: None


### Why `else` exists

Without `else`, you have to put post-success work *after* the `try`, but you might forget that an exception was caught and execution kept going. Or worse: you put it *inside* the `try`, where its own exceptions will be wrongly caught.

In [3]:
# Bad: post-success work inside try — a bug in send() would be hidden as a 'parse error'.
def process_bad(text):
    try:
        data = int(text)
        send_to_api(data)        # <-- if this raises ValueError for ANY reason, we'd report 'parse error'
    except ValueError:
        print("parse error")

# Good: post-success work in else — only runs if the try block was clean.
def process_good(text):
    try:
        data = int(text)
    except ValueError:
        print("parse error")
    else:
        send_to_api(data)         # <-- separate scope; its errors propagate normally

def send_to_api(x):
    print("sent", x)

process_good("42")
process_good("abc")

sent 42
parse error


## Catching several types at once

In [4]:
def lookup(d, key, index):
    try:
        return d[key][index]
    except (KeyError, IndexError) as e:
        return f"missing: {type(e).__name__} -> {e}"

data = {"users": ["alice", "bob"]}
print(lookup(data, "users", 0))     # alice
print(lookup(data, "users", 99))    # IndexError
print(lookup(data, "admins", 0))    # KeyError

alice
missing: IndexError -> list index out of range
missing: KeyError -> 'admins'


## `raise ... from ...` — preserve the original cause

When you re-raise as a different exception, **always** keep the original. Otherwise the traceback hides the real bug.

In [5]:
import json

class ConfigError(Exception):
    """Anything wrong with the config file."""

# BAD — original cause is wiped, debugging is harder.
def load_bad(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        raise ConfigError("bad config")

# GOOD — chain with `from e`. Traceback shows both errors.
def load_good(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ConfigError("bad config") from e

for label, fn in [("BAD ", load_bad), ("GOOD", load_good)]:
    try:
        fn("{not json}")
    except ConfigError as e:
        print(f"{label}: {type(e).__name__}: {e}  |  caused by: {e.__cause__!r}")

BAD : ConfigError: bad config  |  caused by: None
GOOD: ConfigError: bad config  |  caused by: JSONDecodeError('Expecting property name enclosed in double quotes: line 1 column 2 (char 1)')


### `from None` — suppress chaining

Rare, but sometimes you want to *hide* the original exception (e.g. to keep an API surface clean for a library user). Use `raise NewError(...) from None`.

## EAFP vs LBYL — the Python style

- **LBYL** (Look Before You Leap) — check first: `if key in d: return d[key]`.
- **EAFP** (Easier to Ask Forgiveness than Permission) — try and handle: `try: return d[key] except KeyError: ...`.

Python culturally prefers EAFP — it's race-free (LBYL can be wrong by the time the action runs) and often faster on the happy path. But for cheap checks (`if x is None`), LBYL is fine.

In [6]:
d = {"a": 1, "b": 2}

# LBYL
if "a" in d:
    print(d["a"])

# EAFP
try:
    print(d["a"])
except KeyError:
    pass

# In practice, .get() handles this cleanly — no try block needed.
print(d.get("missing", "default"))

1
1
default


## Mini-exercise

1. Write `read_int_from(prompt: str) -> int` that uses `input(prompt)` and re-asks on bad input (loop with `try`/`except ValueError`).
2. Predict the printed order, then run it. Where does `else` fit in?
   ```python
   try:
       print("A"); x = 1 / 0
   except ZeroDivisionError:
       print("B")
   else:
       print("C")
   finally:
       print("D")
   ```
3. Rewrite the snippet above so the division succeeds and observe `C` appear. Confirm `D` runs both times.
4. Why is this a bug?
   ```python
   try:
       process(input)
   except Exception:
       pass
   ```

In [7]:
# your solutions here


**Takeaways**

- Catch **specific** exceptions. `except Exception:` only at the top of the program, with logging.
- `else` runs on success; `finally` runs always. Use `else` for post-success work to keep it out of the `try` scope.
- `raise NewError(...) from e` preserves the original cause in the traceback.
- EAFP is idiomatic Python — try and handle, not check-then-do.

Next: [02_custom_exceptions.ipynb](02_custom_exceptions.ipynb) — design your own.